In [1]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import pandas as pd
import ast
from pprint import pprint
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb, to_hex
from collections import defaultdict
from shapely.geometry import LineString
import networkx as nx

from haversine import haversine, Unit

from leuvenmapmatching.map.inmem import InMemMap
from leuvenmapmatching.matcher.distance import DistanceMatcher

pd.set_option('future.no_silent_downcasting', True)

import time

In [2]:
# Defining Porto's bounding box

north = 41.188771290159195
south = 41.13090363866696
west  = -8.699966476287791
east  = -8.549738123284369


bbox = (west, south, east, north)
# Fetch network from the bounding box
G = ox.graph_from_bbox(
    bbox=bbox,
    network_type="drive",   # can be 'drive', 'walk', etc.
    simplify=False,
)

def make_edges_explicit_oneway(G: nx.MultiDiGraph):
    """
    Update OSMnx graph in-place:
    - Ensure every edge has a 'geometry' attribute (LineString)
    - Convert all edges to explicit one-way edges
    - For two-way edges (oneway=False), add a reverse edge if missing
      with reversed geometry, and set both edges to oneway=True
    """
    edges_to_add = []

    for u, v, key, data in list(G.edges(keys=True, data=True)):
        # 1. Ensure geometry exists
        if 'geometry' not in data:
            u_coord = (G.nodes[u]['y'], G.nodes[u]['x'])
            v_coord = (G.nodes[v]['y'], G.nodes[v]['x'])
            data['geometry'] = LineString([u_coord, v_coord])

        # 2. If edge is two-way, handle reverse edge
        if not data.get('oneway', False):
            # Check if reverse edge exists
            reverse_edges = list(G.get_edge_data(v, u, default={}).values())
            if reverse_edges:
                # Reverse edge exists, set it to oneway=True
                for rev_data in reverse_edges:
                    rev_data['oneway'] = True
            else:
                # Reverse edge missing: create it with reversed geometry
                reversed_geom = LineString(list(data['geometry'].coords)[::-1])
                reversed_data = data.copy()
                reversed_data['geometry'] = reversed_geom
                reversed_data['oneway'] = True
                edges_to_add.append((v, u, reversed_data))

            # Set original edge to oneway=True
            data['oneway'] = True

    # Add reverse edges after the loop
    for u, v, data in edges_to_add:
        G.add_edge(u, v, **data)

    print(f"Graph processed: {G.number_of_edges()} edges in total.")
    print(f"Added {len(edges_to_add)} reverse edges where needed.")


make_edges_explicit_oneway(G)



def create_leuven_map(osmnx_graph):
    """Convert OSMnx graph to LeuvenMapMatching map format"""
    map_converted = InMemMap("my_map", use_latlon=True)
    
    # Add nodes to the map - FIXED: using (lat, lon) order
    for node_id, data in osmnx_graph.nodes(data=True):
        map_converted.add_node(node_id, (data['y'], data['x']))  # (lat, lon)
    
    # Add edges to the map
    for u, v, data in osmnx_graph.edges(data=True):
        map_converted.add_edge(u, v)
        if not data.get('oneway', False):
            map_converted.add_edge(v, u)
    
    node_count = len([i for i in map_converted.all_nodes()])
    edge_count = len([i for i in map_converted.all_edges()])
    print(f"Map created with {node_count} nodes and {edge_count} edges")
    return map_converted
map_leuven = create_leuven_map(G)


Graph processed: 80999 edges in total.
Added 0 reverse edges where needed.
Map created with 51947 nodes and 80999 edges


In [3]:
# Create the smaller dataset

# traj_df = pd.read_csv("train.csv", nrows=15, index_col=0)
# traj_df.to_csv( "train_15.csv")

In [4]:
# Loading the first 15 trajectories
traj_df = pd.read_csv("train_15.csv", index_col=0)

polylines = traj_df["POLYLINE"].to_numpy()
trajectories = []

for line in polylines:
    traj = ast.literal_eval(line)  # list of (lon, lat)
    trajectories.append(traj)


In [5]:
# Vivid colors for trips and routes
vivid_colors = [
    "#e41a1c",  # red
    "#377eb8",  # blue
    "#4daf4a",  # green
    "#984ea3",  # purple
    "#ff7f00",  # orange
    "#ffff33",  # yellow
    "#a65628",  # brown
    "#f781bf",  # pink
    "#999999",  # gray
    "#66c2a5",  # teal
    "#fc8d62",  # salmon
    "#8da0cb",  # light blue
    "#000000",  # black
    "#a6d854",  # lime
    "#ffd92f"   # gold
]


## Unclean Trips

Here I plot the raw trips for part 2 as is, without and data cleansing, note that some trajectories have outliers

In [6]:
# Base graph plot
fig, ax = ox.plot_graph(
    G,
    figsize=(15,9),
    bgcolor="white",
    node_size=6,
    node_color="lightgray",
    node_zorder=1,
    edge_color="lightgray",
    edge_linewidth=0.8,
    show=False,
    close=False,
)

# Plot each trajectory
for index, traj in enumerate(trajectories):
    x, y = zip(*traj)
    color = vivid_colors[index % len(vivid_colors)]  # wrap around if >15 trips
    ax.plot(
        x, y,
        color=color,
        marker="o",
        markersize=4,
        markeredgecolor="black",
        markeredgewidth=0.3,
        linewidth=1.5,
        alpha=0.9,
        zorder=3,
        label=f"Trip {index+1}"
    )

# Legend and title
ax.legend(
    loc="upper right",
    fontsize=8,
    frameon=True,
    facecolor="white",
    edgecolor="black"
)

ax.set_title("First 15 Porto Trips on Road Network", fontsize=14, y=0.95)
fig.savefig("Images/raw_gps_plot_part_2.png", dpi=300, bbox_inches="tight", pad_inches=0)

plt.close(fig)


## Map Matching from unclean trips

The following code block uses ValhallaMatcher to match the raw trips to routes as is,
this leads to spurious routes that will be cleaned later.


In [29]:
# Using Valhalla Matcher to get routes from trips
matcher = DistanceMatcher(
    map_leuven,
    max_dist=300,        # 300 meters maximum distance
    min_prob_norm=0.001,
    obs_noise=50,        # 50 meters GPS noise
    obs_noise_ne=50,
    dist_noise=50,
    use_latlon=True,
    non_emitting_states=True,
    max_lattice_width=20
)

routes = [] # collecion of raw edges



for index, traj in enumerate(trajectories): 
    print(f"---- Processing trips {index+1}/{len(trajectories)} ----")

    # Create Trace directly from GeoDataFrame (in EPSG:4326 for Valhalla)

    # Perform matching
    traj = [(lat, lon) for lon, lat in traj]  # Convert to (lat, lon)
    match_result, final_indx = matcher.match(traj)

    if final_indx + 1 == len(traj):
        print("all gps points allocated")
    else:
        print(f"{len(traj) - final_indx - 1} gps points not allocated")

        
    routes.append(match_result)


Searching closeby nodes with linear search, use an index and set max_dist


---- Processing trips 1/15 ----


Searching closeby nodes with linear search, use an index and set max_dist


all gps points allocated
---- Processing trips 2/15 ----


Searching closeby nodes with linear search, use an index and set max_dist


...

all gps points allocated
---- Processing trips 13/15 ----


Searching closeby nodes with linear search, use an index and set max_dist


all gps points allocated
---- Processing trips 14/15 ----


Searching closeby nodes with linear search, use an index and set max_dist


all gps points allocated
---- Processing trips 15/15 ----
all gps points allocated


### Edge Dict
For each edge store its data

In [30]:
edges = G.edges(data = True) # A tuple of u,v,data convert to a dict of (u,v):data

for u, v, key, data in G.edges(keys=True, data=True):
    # Check 1: geometry must exist
    assert 'geometry' in data, f"Missing geometry on edge ({u}, {v}, key={key})"
    assert data['geometry'] is not None, f"Null geometry on edge ({u}, {v}, key={key})"

    # Check 2: edge must be one-way
    assert data.get('oneway', False) is True, f"Edge ({u}, {v}, key={key}) is not marked as one-way"

print(" All edges verified: every edge has geometry and is marked one-way.")


edges_dict = {(u,v):data for u,v,data in edges}




 All edges verified: every edge has geometry and is marked one-way.


## Plotting Unclean Routes

In [31]:
# Prepare route geometries
routes_edges = []
for route in routes:
    edge_list = [edges_dict[link]['geometry'] for link in route]

    # The coordinates are flipped to accountfor rotation during mapmatching
    yy = [x for edge in edge_list for x in edge.xy[0]]
    xx = [y for edge in edge_list for y in edge.xy[1]]
    routes_edges.append([xx, yy])


# --- Plot base OSM graph ---

fig, ax = ox.plot_graph(
    G,
    figsize=(15,9),
    bgcolor="white",
    node_size=6,
    node_color="lightgray",
    node_zorder=1,
    edge_color="lightgray",
    edge_linewidth=0.8,
    show=False,
    close=False,
)

# --- Plot matched routes ---
for index, (xx, yy) in enumerate(routes_edges):
    color = vivid_colors[index % len(vivid_colors)]
    ax.plot(xx, yy, color=color, linewidth=1.5, alpha=0.9, zorder=3, label=f"Route {index+1}")

# --- Styling ---
ax.set_title("Matched Routes on Road Network", fontsize=14, y=0.95)
ax.legend(loc="upper right", fontsize=8, frameon=True, facecolor="white", edgecolor="black")

# --- Save figure ---
fig.savefig("Images/unclean_routes_part_4.png", dpi=300, bbox_inches="tight", pad_inches=0)
plt.close(fig)


## Cleaning Trajctories and Trips

Plot a particular Trajectory and manually clean it and convert the Trajectories to trips

In [ ]:

particular_traj = trajectories[2]
# compute_distances from previous popints, treat lat as x and long as y

rel_distances = []
for index in range(len(particular_traj[1:])):
    rel_distances.append(haversine(particular_traj[index], particular_traj[index-1], Unit.METERS))

pprint(rel_distances)

# Plot the line
plt.figure(figsize=(10,10))
x, y = zip(*particular_traj)
plt.plot(x, y, '-o', markersize=3, linewidth=1, color='blue')

# Annotate each point with its index
for index, (xi, yi) in enumerate(zip(x, y)):
    plt.text(xi, yi, str(index), fontsize=8, ha='right', va='bottom', color='darkred')

# Styling
plt.axis('equal')
plt.xlabel("X (KM, EPSG:3857)")
plt.ylabel("Y (KM, EPSG:3857)")
plt.grid(True, linestyle='--', alpha=0.3)

plt.show()

In [ ]:
# manually cleanse trajectories by removing outliers
trajectories[2] = [pt for i, pt in enumerate(trajectories[2]) if i not in {34,35}]
trajectories[3] = [pt for i, pt in enumerate(trajectories[3]) if i not in {15,16,17,18}]

# Recreate trips
trips = [  gpd.GeoDataFrame(
        geometry=[Point(lon, lat) for lon, lat in traj],crs="EPSG:4326"  # convert from lat/long
        ).to_crs(porto_crs) 
    for traj in trajectories]

## Map Matching on Cleansed Trips

In [ ]:
routes = []
matches_dfs = []


for index, traj in enumerate(trips): 
    print(f"---- Processing trips {index+1}/{len(trips)} ----")

    # Create Trace directly from GeoDataFrame (in EPSG:4326 for Valhalla)
    trace = Trace.from_geo_dataframe(traj, xy=False)  

    # Perform matching
    match_result = matcher.match_trace(trace)

    # Store per-coordinate matches for inspection
    matches_dfs.append(match_result.matches_to_dataframe()[["coordinate_id", "geom"]])


    path = [ route.geom for route in match_result.path]
    routes.append(path)

    print(f"Matched to {len(routes[-1])} road segments")
    time.sleep(1)

In [ ]:
print(routes[0][0].xy)
print(routes[0][1].xy)
print(routes[0][2].xy)
print(routes[0][3].xy)
print(routes[0][4].xy)
print(routes[0][5].xy)

In [ ]:
"""
Creating route_edges, that store a list of x,y points for each route
and link counts, a dict that stores counts of each link.
"""
routes_edges = [] # This is not used for any analysis just for plotting
link_counts = {} # Used for most frequent road link tahken analysis

for index, route in enumerate(routes): 
    xx, yy = [], [] 
    for link in route: # each link is start, end, key
        try:
            link_counts[link] +=1
        except:
            link_counts[link] = 1

        x, y = link.xy 
        xx.extend(x) 
        yy.extend(y) 
    routes_edges.append([xx, yy])

## Visualizing Cleansed Trips



In [ ]:
# Base graph plot
fig, ax = ox.plot_graph(
    G,
    figsize=(15,9),
    bgcolor="white",
    node_size=6,
    node_color="lightgray",
    node_zorder=1,
    edge_color="lightgray",
    edge_linewidth=0.8,
    show=False,
    close=False,
)

# Plot each trajectory
for index, traj in enumerate(trips):
    x, y = traj.geometry.x, traj.geometry.y
    color = vivid_colors[index % len(vivid_colors)]  # wrap around if >15 trips
    ax.plot(
        x, y,
        color=color,
        marker="o",
        markersize=4,
        markeredgecolor="black",
        markeredgewidth=0.3,
        linewidth=1.5,
        alpha=0.9,
        zorder=3,
        label=f"Trip {index+1}"
    )

# Legend and title
ax.legend(
    loc="upper right",
    fontsize=8,
    frameon=True,
    facecolor="white",
    edgecolor="black"
)

ax.set_title("First 15 Cleansed Trips on Road Network", fontsize=14, y=0.95)
fig.savefig("Images/cleansed_gps_plot_part_2.png", dpi=300, bbox_inches="tight", pad_inches=0)

plt.close(fig)


## Visualizing Cleansed Trips and Routes

In this section, we plot the cleansed trips and their matched routes.
We create multiple of these plots to avoid cluttering. Routes are solid, trips are not


In [ ]:
def lighten_color(color, factor=0.6):
    """Lighten a color by a given factor (0=white, 1=original)."""
    rgb = to_rgb(color)
    light_rgb = [1 - factor*(1-c) for c in rgb]
    return to_hex(light_rgb)
batches = [
    [5,2],
    [1,3,10],
    [8,11,14],
    [12,9,13],
    [6,15,7,4]
]

for batch in batches:
    fig, ax = ox.plot_graph(
        G,
        figsize=(15,9),
        bgcolor="white",
        node_size=6,
        node_color="lightgray",
        node_zorder=1,
        edge_color="lightgray",
        edge_linewidth=3,
        show=False,
        close=False,
    )
    
    batch_str = "-".join(str(b) for b in batch)
    
    # Collect all points for this batch to compute bounding box
    all_x, all_y = [], []
    
    for index in batch:
        index -= 1
        # Base color
        base_color = vivid_colors[index]
        # Slightly lighter for trip
        trip_color = lighten_color(base_color, factor=0.5)
        
        # Plot route (solid, thick) using routes_edges
        x, y = routes_edges[index]
        ax.plot(x, y, color=base_color, linewidth=2, alpha=0.8, zorder=4, label=f"Route {index+1}")
        all_x.extend(x)
        all_y.extend(y)
        
        # Plot trip (dashed, thinner, with markers)
        x_trip, y_trip = trips[index].geometry.x, trips[index].geometry.y
        ax.plot(
            x_trip, y_trip, color=trip_color, linestyle="--", linewidth=1,
            marker="o", markersize=4, markeredgecolor="black", markeredgewidth=0.3,
            alpha=0.9, zorder=5, label=f"Trip {index+1}"
        )
        all_x.extend(x_trip)
        all_y.extend(y_trip)
    
    # Compute bounding box with small padding (5%)
    pad_x = (max(all_x) - min(all_x)) * 0.05
    pad_y = (max(all_y) - min(all_y)) * 0.05
    ax.set_xlim(min(all_x) - pad_x, max(all_x) + pad_x)
    ax.set_ylim(min(all_y) - pad_y, max(all_y) + pad_y)
    
    ax.set_title(f"Trips & Routes {batch_str}", fontsize=14, y=0.95)
    ax.legend(loc="upper right", fontsize=8, frameon=True, facecolor="white", edgecolor="black")
    
    # Save figure
    fig.savefig(f"Images/matched_route_{batch_str}_part_4.png", dpi=300, bbox_inches="tight", pad_inches=0)
    plt.close(fig)


In [ ]:
# linkcounts was calculated in the previous box

top_links = [k for k, _ in sorted(link_counts.items(), key=lambda x: x[1], reverse=True)[:15]]


# --- Plot base OSM graph (limited to bbox) ---
fig, ax = ox.plot_graph(
    G,
    figsize=(15,9),
    bgcolor="white",
    node_size=6,
    node_color="lightgray",
    node_zorder=1,
    edge_color="lightgray",
    edge_linewidth=0.8,
    show=False,
    close=False,
)
# --- Plot top links ---
for index, link in enumerate(top_links):
    x, y = link.xy
    ax.plot(x, y,linewidth=3,alpha=0.8,zorder=4,label=f"Top Link {index+1}")

ax.set_title("Top 15 Most Traversed Links on Road Network", fontsize=14, pad=20)

# optional legend (can be omitted if cluttered)
ax.legend(loc="upper right",fontsize=8,frameon=True,facecolor="white",edgecolor="black")

fig.tight_layout(rect=[0, 0, 1, 0.97], pad=0.5)

# Save & show
fig.savefig("Images/top_links_on_network_part_5_1.png", dpi=300)
plt.close(fig)

In [ ]:
# Container: link_id -> [total_points, total_routes]
link_stats = defaultdict(lambda: [0, 0])
N = 15

# Iterate over routes
for traj_df in matches_dfs:
    # Count GPS points per link in this route
    traj_links = traj_df['geom'].value_counts().astype(float)  # link -> number of GPS points in this traj
    
    print(traj_links)
    # Subtract 0.5 from first and last road_id
    first_link = traj_df.iloc[0]['geom']
    last_link = traj_df.iloc[-1]['geom']
    traj_links[first_link] -= 0.5
    traj_links[last_link] -= 0.5
    
    # Accumulate points and route count
    for link, count in traj_links.items():
        link_stats[link][0] += count       # accumulate GPS points
        link_stats[link][1] += 1           # increment number of trips that traversed this link

# Compute average GPS points per trajectory for each link
avg_points_per_route = {link: total_points/total_routes
                        for link, (total_points, total_routes) in link_stats.items()}

# Get top 10 links with largest average points per route
top_slowest = sorted(avg_points_per_route.items(), key=lambda x: x[1], reverse=True)[:N]

# Display results
for rank, (link, avg_points) in enumerate(top_slowest, start=1):
    print(f"{rank:2d}. Link {link} — average GPS points per trajectory: {avg_points:.2f}")


# top_slowest is a list of tuples: (link_id, avg_points)
top_slowest_geom = []
for link_geom, _ in top_slowest:
    x, y = link_geom.xy
    top_slowest_geom.append([x, y])

# --- Plot base OSM graph (limited to bbox) ---
fig, ax = ox.plot_graph(G,bbox=bbox,figsize=(15, 8),node_size=6,node_color="white",edge_color="lightgray",edge_linewidth=0.8,show=False,close=False,)

# --- Plot top slowest links ---
for index, link_geom in enumerate(top_slowest_geom):
    x, y = link_geom
    ax.plot( x, y,linewidth=3,alpha=0.8,zorder=4,label=f"Slow Link {index+1}")

# --- Styling ---
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.set_title("Top 10 Slowest Links on Road Network (avg GPS points per route)", fontsize=14, pad=20)

ax.set_xlim(all_x.min() - pad_x, all_x.max() + pad_x)
ax.set_ylim(all_y.min() - pad_y, all_y.max() + pad_y)

# optional legend
ax.legend(loc="upper right",fontsize=8,frameon=True,facecolor="white",edgecolor="black")

fig.tight_layout(rect=[0, 0, 1, 0.97], pad=0.5)

# Save & show
fig.savefig("Images/top_slowest_links_part_5_2.png", dpi=300)
plt.close(fig)

